In [85]:
# @title Setup
# params to modify
# queries = ['Driver: San Francisco']
queries = ['Driver: San Francisco', 'Driver: San Francisco review', 
           'Driver: San Francisco retrospective', 
           'Driver: San Francisco critique', 'Driver: San Francisco analysis']

# excludes = ["part", "episode", '#', 'short']
# queries = [f'{query} -{' -'.join(excludes)}' for query in queries]

cadence = {'years': 10}

from datetime import datetime, timedelta
import pandas as pd

start_date = (datetime.now() - pd.DateOffset(**cadence)).strftime("%Y-%m-%d")
end_date = datetime.now().strftime("%Y-%m-%d")

# cadence term
key, val = list(cadence.items())[0]
cadence_str =  str(key)[0] + str(val)

# create db anem
queries = sorted(queries)

# Get API keys
API_KEYS = """YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY
YOUR_YOUTUBE_API_KEY""".splitlines()

In [86]:
# @title Parallel Search

# import tqdm, build, HttpError
from tqdm.notebook import tqdm
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

tqdm.pandas()

keys = API_KEYS.copy()

def get_videos_for_period(query_string, start_date, pbar, cadence=None):

    next_page_token = None
    
    period = {key: 1 for key in cadence.keys()}    
    
    start_date = pd.Timestamp(start_date).strftime('%Y-%m-%dT00:00:00Z')
    end_date = (
        pd.Timestamp(start_date) + pd.DateOffset(**period)
        if cadence else datetime.now()
    ).strftime('%Y-%m-%dT00:00:00Z')       
    
    search_results_data = []
    while True:
        
        youtube = build('youtube', 'v3', developerKey=keys[0])
        # First API request (search)
        request = youtube.search().list(
            part='snippet',
            maxResults=50,
            pageToken=next_page_token,
            publishedAfter=start_date,
            publishedBefore=end_date,
            q=query_string,
            relevanceLanguage='en',
            type='video',
            videoCategoryId='20',
            # videoCaption='closedCaption'
        )
        
        try:
            response = request.execute()
        except HttpError as e:
            
            if e.resp.status == 403:
                
                pbar.set_postfix_str(f"Keys remaining: {len(keys)}")
                
                keys.pop(0)
                
                continue
        
        next_page_token = response.get('nextPageToken')
        search_items = response.get('items')
        search_results_data.extend(search_items)

        if not next_page_token:
            break
        
    return search_results_data

from itertools import product
from concurrent.futures import ThreadPoolExecutor, as_completed
import polars as pl

# Function to parallelize the queries and date range processing
def parallel_search_query_execution(queries, period_start_dates, cadence):
    
    # Combine the queries and dates into a product list
    queries_and_dates = list(product(queries, period_start_dates))

    # Initialize progress bar and list for storing results
    pbar = tqdm(total=len(queries_and_dates), desc="Processing Queries and Dates")
    search_data = []

    # Use ThreadPoolExecutor to handle parallel processing
    with ThreadPoolExecutor(max_workers=4) as executor:  # Adjust max_workers as needed
        futures = []
        
        # Submit all query-date tasks for parallel execution
        for query, start_date in queries_and_dates:
            futures.append(executor.submit(get_videos_for_period, query, start_date, pbar, cadence))

        # Collect results as they are completed
        for future in as_completed(futures):
            video_data = future.result()
            search_data.append(video_data)
            pbar.update(1)

    search_results = [elem for sub in search_data for elem in sub]
    
    search_results_df = pl.json_normalize(search_results)

    return search_results_df


# Calculate number of periods
num_periods = list(cadence.values())[0] + 1
period_start_dates = pd.date_range(start=start_date, end=end_date, periods=num_periods)[:-1]

# Call the parallelized search execution function
search_results_df = parallel_search_query_execution(queries, period_start_dates, cadence)

search_results_df.shape

Processing Queries and Dates:   0%|          | 0/50 [00:00<?, ?it/s]

(19025, 20)

In [87]:
# @title Video Statistics

def fetch_video_statistics_for_id_string(id_string):
    
    try:
        youtube = build('youtube', 'v3', developerKey=api_keys[0])

        request = youtube.videos().list(
            part='contentDetails,id,liveStreamingDetails,snippet,statistics,topicDetails',
            id=id_string
        )
        
        response = request.execute()
        
        items = response.get('items')
        
        return items
        
    except HttpError as e:
        
        if e.resp.status == 403:  # Quota exceeded
            api_keys.pop(0)
        else:
            raise  # Re-raise the exception if it's not a quota error
        
        
api_keys = API_KEYS.copy()

# break into groups of 50
video_ids = search_results_df['id.videoId'].to_numpy().flatten()
batch_size = 50
video_id_batches = [video_ids[i:i + batch_size] for i in range(0, len(video_ids), batch_size)]

video_id_strings = [','.join(video_id_batch) for video_id_batch in video_id_batches]
video_statistics_data = [fetch_video_statistics_for_id_string(video_id_string) for video_id_string in tqdm(video_id_strings)]

no_none = [item for item in video_statistics_data if item]
flattened_video_statistics_data = [item for sublist in no_none for item in sublist]
video_statistics_df = pl.json_normalize(flattened_video_statistics_data)

video_statistics_df.shape

  0%|          | 0/381 [00:00<?, ?it/s]

(18975, 41)

In [88]:
# @title get game data
import logging
import asyncio
import aiohttp
import json
import random
import pandas as pd
from tqdm.notebook import tqdm
import nest_asyncio

async def fetch_data(session, video_ids_batch, part_str, timeout_seconds=256, max_retries=64):
    """Fetch data for a batch of video IDs"""
    url = f'http://localhost:8080/videos?part={part_str}&id={video_ids_batch}'
    
    for attempt in range(max_retries):
        try:
            async with session.get(url, timeout=aiohttp.ClientTimeout(total=timeout_seconds)) as response:
                if response.status == 429:
                    delay = min(2 ** attempt + random.uniform(0, 1), 60)
                    await asyncio.sleep(delay)
                    continue
                
                response.raise_for_status()
                text_response = await response.text()
                
                json_start = text_response.find('{')
                if json_start != -1:
                    try:
                        json_data = json.loads(text_response[json_start:])
                        return json_data.get('items', [])
                    except json.JSONDecodeError as e:
                        logging.error(f"JSON decode error: {e}")
                        await asyncio.sleep(2 ** attempt)
                return []
        
        except Exception as e:
            logging.error(f"Error fetching data: {type(e).__name__}: {str(e)}")
            await asyncio.sleep(2 ** attempt)
    
    logging.error(f"Failed to fetch data for batch: {video_ids_batch}")
    return []

async def fetch_all_videos(video_ids, parts=['mostReplayed', 'chapters', 'activity'], 
                         batch_size=1, concurrent_requests=64):
    """Fetch all video data and return as a list"""
    part_str = ','.join(parts)
    
    # # Create batches
    # batches = [','.join(video_ids[i:i + batch_size]) 
    #            for i in range(0, len(video_ids), batch_size)]
    
    connector = aiohttp.TCPConnector(limit=concurrent_requests, force_close=True)
    all_items = []
    
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = []
        for batch in video_ids:
            task = fetch_data(session, batch, part_str)
            tasks.append(task)
        
        # Process all batches with progress bar
        for results in tqdm(asyncio.as_completed(tasks), total=len(tasks), 
                          desc="Fetching video data"):
            batch_items = await results
            all_items.extend(batch_items)
    
    return all_items

def process_video_data(items):
    """Convert API response items into a DataFrame"""
    if not items:
        return pd.DataFrame()
    
    # Flatten nested JSON structure
    flattened_data = []
    for item in items:
        flat_item = {}
        for key, value in item.items():
            if isinstance(value, dict):
                for sub_key, sub_value in value.items():
                    flat_item[f"{key}_{sub_key}"] = sub_value
            else:
                flat_item[key] = value
        flattened_data.append(flat_item)
    
    return pd.DataFrame(flattened_data)

async def main(data_df):
    """Main function to fetch and process video data"""
    # Extract unique video IDs
    video_ids = list(set(data_df['id']))
    
    # Fetch all video data
    items = await fetch_all_videos(video_ids)
    
    # Process into DataFrame
    result_df = process_video_data(items)
    
    return result_df

# Usage
nest_asyncio.apply()

async def get_video_data(input_df):
    return await main(input_df)

# For a single run
result_df = await get_video_data(video_statistics_df)

result_df.shape

Fetching video data:   0%|          | 0/12182 [00:00<?, ?it/s]

ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Error fetching data: TimeoutError: 
ERROR:root:Er

(12182, 12)

In [92]:
# @title Join and Filter before Transcripts
game_df = result_df.copy()
# game_df = game_df[game_df['activity_name']=='Driver: San Francisco']
game_df = game_df.set_index('id')
game_df = game_df.drop(columns=['kind', 'etag'])

full_df = game_df.join(video_statistics_df.to_pandas().set_index('id'), how='inner').reset_index()
full_df = full_df.drop(columns=['kind', 'etag'])
# full_df = full_df.set_index('id')
full_df = pl.DataFrame(full_df)

languages = ['en', 'en-US', 'en-GB', 'en-IN', 'zxx', 'en-CA', 'en-AU', 'null', None]

full_df = full_df.filter(
    pl.col('contentDetails.dimension') == '2d',
    pl.col('contentDetails.definition') == 'hd',
    pl.col('contentDetails.projection') == 'rectangular',
    pl.col('snippet.categoryId') == '20',
    pl.col('snippet.liveBroadcastContent') == 'none',
    pl.col('snippet.defaultAudioLanguage').is_in(languages) | pl.col('snippet.defaultAudioLanguage').is_null(),
    # pl.col('snippet.defaultLanguage').is_in(languages) | pl.col('snippet.defaultLanguage').is_null(),
    # pl.col('liveStreamingDetails.actualStartTime').is_null(),
    # pl.col('liveStreamingDetails.actualEndTime').is_null(),
    # pl.col('liveStreamingDetails.scheduledStartTime').is_null(),
)

full_df = full_df.with_columns(
    pl.col('statistics.viewCount').cast(pl.Int32)
)

full_df = full_df.sort('statistics.viewCount')

import re

# Function to parse ISO 8601 durations into seconds
def parse_duration_iso8601(duration):
    match = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", duration)
    hours = int(match.group(1) or 0)
    minutes = int(match.group(2) or 0)
    seconds = int(match.group(3) or 0)
    return hours * 3600 + minutes * 60 + seconds

full_df = full_df.with_columns(
    pl.col('contentDetails.duration')
    .map_elements(parse_duration_iso8601, return_dtype=pl.Int64)
).filter(pl.col("contentDetails.duration") > 61)

full_df = full_df.to_pandas()
full_df = full_df.drop_duplicates(subset=['id']).reset_index(drop=True)
full_df = pl.DataFrame(full_df)

full_df.shape

(9459, 48)

In [ ]:
# @title Get Transcripts

video_ids = full_df.select('id').to_series().to_list()

from youtube_transcript_api import (
    YouTubeTranscriptApi,
    NoTranscriptFound,
    TranscriptsDisabled
    )

from xml.etree.ElementTree import ParseError

from tqdm.notebook import tqdm

transcripts_data = []

for video_id in tqdm(video_ids):

    transcript_data = {
        'video_id': video_id,
        'language_code': None,
        'is_generated': None,
        'transcript': None
    }

    try:
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
    except TranscriptsDisabled:
        transcripts_data.append(transcript_data)
        continue

    try:
        transcript = transcript_list.find_manually_created_transcript(['en'])
    except NoTranscriptFound:
        try:
            transcript = transcript_list.find_generated_transcript(['en'])
        except NoTranscriptFound:
            transcripts_data.append(transcript_data)
            continue

    try:
        fetched_transcript = transcript.fetch()
    except ParseError:
        transcripts_data.append(transcript_data)
        continue

    transcript_data = {
        'video_id': video_id,
        'language_code': transcript.language_code,
        'is_generated': transcript.is_generated,
        'transcript': fetched_transcript
    }

    transcripts_data.append(transcript_data)

transcripts = []
for transcript in tqdm(transcripts_data):
    video_id = transcript['video_id']
    language_code = transcript['language_code']
    is_generated = transcript['is_generated']
    transcript_df = pl.DataFrame(transcript['transcript'])
    if transcript_df.shape[0] > 0:
        full_text = ' '.join(transcript_df['text'])
        word_count = len(full_text.split(' '))
        transcripts.append({'id': video_id, 'lang': language_code, 'is_gen': is_generated, 'text': full_text, 'word_count': word_count})
transcript_df = pl.DataFrame(transcripts)
transcript_df.shape
 

  0%|          | 0/9459 [00:00<?, ?it/s]

In [ ]:
df = full_df.join(transcript_df, on='id', how='inner')


keep_columns = ['id', 'snippet.publishedAt', 'snippet.title', 'snippet.description', 'snippet.tags', 'snippet.channelTitle',
                'contentDetails.duration', 'statistics.viewCount', 'topicDetails.topicCategories', 'lang', 'is_gen', 'text', 'word_count']

df = df.select(keep_columns)

df = df.with_columns(
    (60 * pl.col('word_count') / pl.col('contentDetails.duration')).alias('wpm'),
)

df = df.sort('wpm')


# df = df.filter(
#     pl.col('contentDetails.duration') > 1440,
#     pl.col('wpm') > 140
# )

df

id,snippet.publishedAt,snippet.title,snippet.description,snippet.tags,snippet.channelTitle,contentDetails.duration,statistics.viewCount,topicDetails.topicCategories,lang,is_gen,text,word_count,wpm
str,str,str,str,list[str],str,i64,i32,list[str],str,bool,str,i64,f64
"""wZe1MOrbBRg""","""2023-03-12T11:07:21Z""","""Driver: San Francisco - Police…","""Driver 1 but taking place in S…","[""driver san francisco"", ""driver san francisco police"", … ""driver san francisco gameplay""]","""Thekillergreece""",667,11656,"[""https://en.wikipedia.org/wiki/Action-adventure_game"", ""https://en.wikipedia.org/wiki/Action_game"", … ""https://en.wikipedia.org/wiki/Video_game_culture""]","""en""",false,"""There are on-screen subtitles.…",5,0.449775
"""YzAo3c1_REU""","""2023-03-18T17:00:27Z""","""Driver San Francisco All Cars …","""Driver San Francisco All Vehic…","[""Gaming"", ""Game"", … ""All Cars Sounds""]","""snyboyza""",7511,21602,"[""https://en.wikipedia.org/wiki/Action_game"", ""https://en.wikipedia.org/wiki/Racing_video_game"", ""https://en.wikipedia.org/wiki/Video_game_culture""]","""en""",false,"""Note : Some Modified or Varian…",282,2.252696
"""22qIFtavMmU""","""2022-05-12T12:29:08Z""","""Evolution of DRIVER Games 1999…","""Includes Gameplay From ALL Dri…","[""evolution of driver games 1999-2021"", ""Driver Games"", … ""ubisoft""]","""Game Evolutions""",535,813009,"[""https://en.wikipedia.org/wiki/Action-adventure_game"", ""https://en.wikipedia.org/wiki/Action_game"", … ""https://en.wikipedia.org/wiki/Video_game_culture""]","""en""",true,"""driver you are the wheel man 1…",144,16.149533
"""5oFWKtKzPj4""","""2023-03-18T17:00:25Z""","""Driver: San Francisco - All Ca…","""Driver San Francisco All Vehic…","[""Gaming"", ""Game"", … ""Driver: San Francisco""]","""snyboyza""",501,52220,"[""https://en.wikipedia.org/wiki/Action-adventure_game"", ""https://en.wikipedia.org/wiki/Action_game"", … ""https://en.wikipedia.org/wiki/Video_game_culture""]","""en""",false,"""(All ASYM) 2006 Lincoln Zephyr…",162,19.401198
"""6gL1iA5ouVE""","""2021-09-17T16:00:04Z""","""Driver San Francisco| Multipla…","""Highlights from playing DSF mu…","[""driver"", ""driver san francisco"", … ""funny""]","""MineAndDrive""",490,9095,"[""https://en.wikipedia.org/wiki/Action-adventure_game"", ""https://en.wikipedia.org/wiki/Action_game"", … ""https://en.wikipedia.org/wiki/Video_game_culture""]","""en""",true,"""[Music] [Music] oh [Music] [Mu…",175,21.428571
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FjOBfsZDNJw""","""2017-12-04T15:00:01Z""","""Driver San Francisco Review: I…","""Welcome to another Gaming Revi…","[""Gaming Reviews""]","""Jack's Gaming""",644,249,"[""https://en.wikipedia.org/wiki/Action-adventure_game"", ""https://en.wikipedia.org/wiki/Action_game"", … ""https://en.wikipedia.org/wiki/Video_game_culture""]","""en""",true,"""hello and welcome to another g…",2253,209.906832
"""JogOqTG_UUs""","""2023-05-21T17:00:08Z""","""8 Things Driver SF Does BETTER…","""From the Great Storyline to th…","[""driver san francisco"", ""driver"", … ""better than fh5""]","""Yuwan Thayakaran""",689,41186,"[""https://en.wikipedia.org/wiki/Action-adventure_game"", ""https://en.wikipedia.org/wiki/Action_game"", … ""https://en.wikipedia.org/wiki/Video_game_culture""]","""en""",true,"""hello it is I now if you asked…",2456,213.875181
"""b8dAx5_cju8""","""2022-06-30T14:21:47Z""","""Driver: San Francisco - Review…","""In the Bonus Points Podcast Jo…","[""Bonus Points"", ""Podcast"", … ""Gaming""]","""The Bonus Points""",3786,3522,"[""https://en.wikipedia.org/wiki/Action-adventure_game"", ""https://en.wikipedia.org/wiki/Action_game"", … ""https://en.wikipedia.org/wiki/Video_game_culture""]","""en""",true,"""[Music] hello and welcome to a…",13731,217.606973


In [ ]:
df.write_parquet('driversf.parquet')

In [1]:
import numpy as np

# plot wpm and view count
import matplotlib.pyplot as plt

temp_df = df.clone().to_pandas()

# temp_df = temp_df[temp_df['wpm'] > 140]
# temp_df = temp_df[temp_df['wpm'] < 200]

temp_df = temp_df.dropna(subset=['wpm', 'statistics.viewCount'])
temp_df = temp_df[np.isfinite(temp_df['statistics.viewCount'])]
temp_df = temp_df[np.isfinite(temp_df['wpm'])]
temp_df = temp_df[temp_df['statistics.viewCount'] > 0]  # Keep only positive view counts

fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(temp_df['wpm'], temp_df['statistics.viewCount'])
ax.set_xlabel('Words per Minute')
ax.set_ylabel('View Count')

# don't use scientific notation
ax.ticklabel_format(style='plain', axis='y')
ax.ticklabel_format(style='plain', axis='x')

# use log scale on y
ax.set_yscale('log')

# add trendline
import numpy as np
from sklearn.linear_model import LinearRegression

X = temp_df['wpm'].to_numpy().reshape(-1, 1)
y = np.log(temp_df['statistics.viewCount'].to_numpy())

reg = LinearRegression()

reg.fit(X, y)

y_pred = np.exp(reg.predict(X))

ax.plot(X, y_pred, color='red')

# add polynomial trendline
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=6)
X_poly = poly.fit_transform(X)

reg_poly = LinearRegression()
reg_poly.fit(X_poly, y)

y_pred_poly = np.exp(reg_poly.predict(X_poly))

ax.plot(X, y_pred_poly, color='green')


plt.show()

NameError: name 'df' is not defined

In [3]:
search_results_df

kind,etag,id.kind,id.videoId,snippet.publishedAt,snippet.channelId,snippet.title,snippet.description,snippet.thumbnails.default.url,snippet.thumbnails.default.width,snippet.thumbnails.default.height,snippet.thumbnails.medium.url,snippet.thumbnails.medium.width,snippet.thumbnails.medium.height,snippet.thumbnails.high.url,snippet.thumbnails.high.width,snippet.thumbnails.high.height,snippet.channelTitle,snippet.liveBroadcastContent,snippet.publishTime
str,str,str,str,str,str,str,str,str,i64,i64,str,i64,i64,str,i64,i64,str,str,str
"""youtube#searchResult""","""yfWSHBr8vrKbZrIq5p8Gdl8R_1Q""","""youtube#video""","""zL7ZPuDAZAk""","""2016-06-05T22:13:17Z""","""UCLsYJRmL3TjaNLQKvJPk_Og""","""Driver: San Francisco SMALL DR…","""Another test :D.""","""https://i.ytimg.com/vi/zL7ZPuD…",120,90,"""https://i.ytimg.com/vi/zL7ZPuD…",320,180,"""https://i.ytimg.com/vi/zL7ZPuD…",480,360,"""LetsRandom77""","""none""","""2016-06-05T22:13:17Z"""
"""youtube#searchResult""","""G5kmbbGVDZTRuwXCb54b36pqXHY""","""youtube#video""","""c7GfB-TZryk""","""2016-07-10T21:10:23Z""","""UCerFo7kkKH_wywerv6A77CA""","""Driver San Francisco │ PLS LOS…","""EPISODE 12! In this video we w…","""https://i.ytimg.com/vi/c7GfB-T…",120,90,"""https://i.ytimg.com/vi/c7GfB-T…",320,180,"""https://i.ytimg.com/vi/c7GfB-T…",480,360,"""SkiStam""","""none""","""2016-07-10T21:10:23Z"""
"""youtube#searchResult""","""55JZLonYQbLH3fx4LHWGfBTwE8E""","""youtube#video""","""MEN1i3A-qQE""","""2016-08-20T23:54:15Z""","""UCMNQvPrWF2C1iPmP4tFVwpQ""","""Watch Dogs 2 - Driver San Fran…","""Watch Dogs 2 News & Informatio…","""https://i.ytimg.com/vi/MEN1i3A…",120,90,"""https://i.ytimg.com/vi/MEN1i3A…",320,180,"""https://i.ytimg.com/vi/MEN1i3A…",480,360,"""UbiCentral""","""none""","""2016-08-20T23:54:15Z"""
"""youtube#searchResult""","""xdl4_lHlJLlFyoGY_yntoehew80""","""youtube#video""","""rlQVpzXN09E""","""2016-06-01T21:49:28Z""","""UCerFo7kkKH_wywerv6A77CA""","""Driver San Francisco │ MORE KI…","""EPISODE 8! Let's stop bombs, l…","""https://i.ytimg.com/vi/rlQVpzX…",120,90,"""https://i.ytimg.com/vi/rlQVpzX…",320,180,"""https://i.ytimg.com/vi/rlQVpzX…",480,360,"""SkiStam""","""none""","""2016-06-01T21:49:28Z"""
"""youtube#searchResult""","""UcKetn24jiF42kLrTJ5vVlOXt8I""","""youtube#video""","""ICwi6oLVxic""","""2016-03-07T15:26:57Z""","""UCyQlzztAdOf5p1l0hTgURvQ""","""Drifting in Driver San Francis…","""""","""https://i.ytimg.com/vi/ICwi6oL…",120,90,"""https://i.ytimg.com/vi/ICwi6oL…",320,180,"""https://i.ytimg.com/vi/ICwi6oL…",480,360,"""Alan Loewen""","""none""","""2016-03-07T15:26:57Z"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""youtube#searchResult""","""hOtpVVDqU0tnf215vp-ktSst6CI""","""youtube#video""","""q-biTMh4ofM""","""2023-07-20T11:45:25Z""","""UCAbHcM41hzH9lku_3XqFYZg""","""AYANEO GEEK 1S Review - A Powe…","""AYANEO GEEK 1S Indiegogo - htt…","""https://i.ytimg.com/vi/q-biTMh…",120,90,"""https://i.ytimg.com/vi/q-biTMh…",320,180,"""https://i.ytimg.com/vi/q-biTMh…",480,360,"""RoeTaKa""","""none""","""2023-07-20T11:45:25Z"""
"""youtube#searchResult""","""jsPexP8TEvT4hv86vgh4wCvUOTs""","""youtube#video""","""oIhvkKsrlp8""","""2023-09-08T04:00:01Z""","""UCuVPpxrm2VAgpH3Ktln4HXg""","""Memento""","""A man with short-term memory l…","""https://i.ytimg.com/vi/oIhvkKs…",120,90,"""https://i.ytimg.com/vi/oIhvkKs…",320,180,"""https://i.ytimg.com/vi/oIhvkKs…",480,360,"""YouTube Movies""","""none""","""2023-09-08T04:00:01Z"""
"""youtube#searchResult""","""t-p6nolhMS0Qp92iERAjIjzq6bI""","""youtube#video""","""gZeuiMkhHT0""","""2023-01-05T18:07:27Z""","""UCJRLyQRpRQCJTRoV7oB2eHA""","""Game Hunting in MELBOURNE, SYD…","""Special thanks to The Gamesmen…","""https://i.ytimg.com/vi/gZeuiMk…",120,90,"""https://i.ytimg.com/vi/gZeuiMk…",320,180,"""https://i.ytimg.com/vi/gZeuiMk…",480,360,"""ColourShedProductions""","""none""","""2023-01-05T18:07:27Z"""
